In [2]:
# Tyler Buck
# CS340 Project Two
# This project is meant to provide Grazioso Salvare with an interactive dashboard
# for viewing and filtering Austin Animal Center animal outcome records stored in
# MongoDB. The application uses a reusable CRUD Python module to query the database,
# display the results in a Dash DataTable, and update a pie chart and geolocation
# map based on the current filter selection and selected table row. The dashboard
# supports the required rescue categories of Water Rescue, Mountain or Wilderness
# Rescue, Disaster or Individual Tracking, and Reset.


from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

###########################
# Data Manipulation / Model
###########################

from CRUD_Python_Module import AnimalShelter

username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
shelter = AnimalShelter(username, password)

# class read method
df = pd.DataFrame.from_records(shelter.read({}))

if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

print("Number of records loaded:", len(df))
print(df.head())


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Center(
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '200px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        )
    ),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center(html.H3('Tyler Buck')),
    html.Hr(),
    html.Div([
        html.Label("Filter by Rescue Type:"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset', 'value': 'reset'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ]),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single",
        row_deletable=False,
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that chart and geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

def get_filter_query(filter_type):
    if filter_type == 'water':
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "$or": [
                {"breed": {"$regex": "Labrador Retriever", "$options": "i"}},
                {"breed": {"$regex": "Chesapeake Bay Retriever", "$options": "i"}},
                {"breed": {"$regex": "Newfoundland", "$options": "i"}}
            ]
        }

    elif filter_type == 'mountain':
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "$or": [
                {"breed": {"$regex": "German Shepherd", "$options": "i"}},
                {"breed": {"$regex": "Alaskan Malamute", "$options": "i"}},
                {"breed": {"$regex": "Old English Sheepdog", "$options": "i"}},
                {"breed": {"$regex": "Siberian Husky", "$options": "i"}},
                {"breed": {"$regex": "Rottweiler", "$options": "i"}}
            ]
        }

    elif filter_type == 'disaster':
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
            "$or": [
                {"breed": {"$regex": "Doberman Pinscher", "$options": "i"}},
                {"breed": {"$regex": "German Shepherd", "$options": "i"}},
                {"breed": {"$regex": "Golden Retriever", "$options": "i"}},
                {"breed": {"$regex": "Bloodhound", "$options": "i"}},
                {"breed": {"$regex": "Rottweiler", "$options": "i"}}
            ]
        }

    else:
        return {}


@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    query = get_filter_query(filter_type)
    dff = pd.DataFrame.from_records(shelter.read(query))

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    if dff.empty:
        return []

    return dff.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [
            html.Div("No chart data available.")
        ]

    dff = pd.DataFrame.from_dict(viewData)

    if 'breed' not in dff.columns or dff.empty:
        return [
            html.Div("No breed data available.")
        ]

    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']

    top_breeds = breed_counts.head(10).copy()

    if len(breed_counts) > 10:
        other_count = breed_counts['count'][10:].sum()
        other_row = pd.DataFrame([{'breed': 'Other', 'count': other_count}])
        top_breeds = pd.concat([top_breeds, other_row], ignore_index=True)

    return [
        dcc.Graph(
            figure=px.pie(
                top_breeds,
                names='breed',
                values='count',
                title='Top Breeds in Current View'
            )
        )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        selected_columns = []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry

@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return [
            html.Div("No map data available.")
        ]

    dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0

    lat = dff.iloc[row]["location_lat"]
    lon = dff.iloc[row]["location_long"]
    breed = dff.iloc[row]["breed"]
    name = dff.iloc[row]["name"]

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[lat, lon],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    icon={
                        "iconUrl": "https://unpkg.com/leaflet@1.9.4/dist/images/marker-icon.png",
                        "shadowUrl": "https://unpkg.com/leaflet@1.9.4/dist/images/marker-shadow.png",
                        "iconSize": [25, 41],
                        "iconAnchor": [12, 41],
                        "popupAnchor": [1, -34],
                        "shadowSize": [41, 41]
                    },
                    children=[
                        dl.Tooltip(str(breed)),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(name))
                        ])
                    ]
                )
            ]
        )
    ]

app.run_server(mode='external', debug=False, port=8060)

Number of records loaded: 10000
   rec_num age_upon_outcome animal_id animal_type                   breed   
0        1          3 years   A746874         Cat  Domestic Shorthair Mix  \
1        9          3 years   A720214         Dog  Labrador Retriever Mix   
2       10         3 months   A664290         Cat  Domestic Shorthair Mix   
3       11           1 year   A721199         Dog  Dachshund Wirehair Mix   
4        2           1 year   A725717         Cat  Domestic Shorthair Mix   

          color date_of_birth             datetime            monthyear   
0   Black/White    2014-04-10  2017-04-11 09:00:00  2017-04-11T09:00:00  \
1     Red/White    2013-02-04  2016-02-11 12:41:00  2016-02-11T12:41:00   
2        Tortie    2013-09-01  2013-12-08 14:58:00  2013-12-08T14:58:00   
3     Tan/White    2015-02-23  2016-02-27 17:49:00  2016-02-27T17:49:00   
4  Silver Tabby    2015-05-02  2016-05-06 10:49:00  2016-05-06T10:49:00   

       name outcome_subtype outcome_type sex_upon_outc

 * Running on http://127.0.0.1:8060/ (Press CTRL+C to quit)
127.0.0.1 - - [15/Apr/2026 17:46:33] "GET /_alive_c5d999eb-01b0-4168-a91d-6c6ce3de7e7b HTTP/1.1" 200 -


Dash app running on https://mediacloud-koalastrong-3000.codio.io/proxy/8060/


127.0.0.1 - - [15/Apr/2026 17:47:28] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [15/Apr/2026 17:47:28] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [15/Apr/2026 17:47:28] "GET /_favicon.ico?v=2.8.1 HTTP/1.1" 200 -
127.0.0.1 - - [15/Apr/2026 17:47:29] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [15/Apr/2026 17:47:29] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [15/Apr/2026 17:47:29] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [15/Apr/2026 17:47:29] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [15/Apr/2026 17:47:29] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [15/Apr/2026 17:47:29] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [15/Apr/2026 17:47:30] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [15/Apr/2026 17:47:30] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [15/Apr/2026 17:47:32] "POST /_dash-update-componen